# 04 — Documenting where and when each data source was observed

The dissertation combines transport accessibility, building records, satellite products, aerial imagery and Street View. These sources do not necessarily describe London in the same calendar year. This notebook records the spatial and temporal meaning of each target and representation so that the modelling results can be interpreted without implying false same-year alignment.

Source dates are taken from extraction code or supplier metadata rather than file creation times. Where capture dates are unavailable, they remain explicitly unresolved.

## Summary of the output

The London-wide aerial collection contains 1,935 JPG/XML tile pairs at 25-centimetre resolution, acquired in 2021–2022; all 26,597 DINOv2 crops can be linked to source-tile dates. TESSERA and AlphaEarth use 2024 observations, while the PTAL outcome refers to 2023.

The retained EPC certificates span 2012–2026, with a median retained year of 2020, so EPC represents a latest-available register snapshot rather than a single-year measure. No reliable capture date is present in the available Street View metadata or sampled EXIF information. The study is therefore described as a multi-temporal cross-sectional comparison of London rather than a strictly same-year analysis.

In [ ]:
# Connect Google Drive and load packages for code, XML and image-metadata inspection.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import re
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({
    name: getattr(_config, name)
    for name in dir(_config)
    if not name.startswith("_")
})

DIGIMAP_DIR = RAW_DIR / "Digimap"
STREETVIEW_DIR = RAW_DIR / "Street_view"
LEGACY_WORKBOOK = CODE_DIR / "Notebook_June_5&24.ipynb"
EPC_FINAL_SUMMARY = AUDIT_DIR / "epc_final_audit_summary.json"

required = {
    "common sample": COMMON_SAMPLE_FINAL_PATH,
    "legacy extraction notebook": LEGACY_WORKBOOK,
    "Digimap 25 cm root": DIGIMAP_DIR,
    "Street View root": STREETVIEW_DIR,
    "EPC final audit": EPC_FINAL_SUMMARY,
    "TESSERA embeddings": TESSERA_EMB_PATH,
    "AlphaEarth final": ALPHA_FINAL_PATH,
    "DINOv2 embeddings": DINO2_EMB_PATH,
    "Street View embeddings": STREET_EMB_PATH,
    "SatCLIP embeddings": SATCLIP_EMB_PATH,
}
for name, path in required.items():
    print(f"{name:28s} | {path.exists()} | {path}")

missing_required = [name for name, path in required.items() if not path.exists()]
if missing_required:
    raise FileNotFoundError(f"Missing required project items: {missing_required}")

## 1. Recover settings from the original extraction code

The extraction notebooks provide the most direct record of parameters supplied to TESSERA and SatCLIP. This section reads those notebooks and extracts the relevant observation-year settings instead of inferring dates from the timestamps of output files.

In [ ]:
# Recover observation-year settings from the original extraction notebook.
legacy_nb = json.loads(LEGACY_WORKBOOK.read_text(encoding="utf-8"))

all_source = "\n".join(
    "".join(cell.get("source", []))
    for cell in legacy_nb.get("cells", [])
)

# TESSERA: extract the year passed to GeoTessera.
tessera_year_matches = re.findall(
    r"sample_embeddings_at_points\s*\([^)]*?year\s*=\s*(\d{4})",
    all_source,
    flags=re.I | re.S
)
tessera_years = sorted(set(int(y) for y in tessera_year_matches))

# Fall back to a more local pattern if formatting prevented the first regex.
if not tessera_years:
    for m in re.finditer(r"sample_embeddings_at_points", all_source, flags=re.I):
        snippet = all_source[m.start():m.start()+800]
        yrs = re.findall(r"year\s*=\s*(\d{4})", snippet)
        tessera_years.extend(int(y) for y in yrs)
    tessera_years = sorted(set(tessera_years))

print("TESSERA years explicitly found in extraction code:", tessera_years)
assert tessera_years == [2024], (
    "Expected the existing TESSERA common-sample embeddings to have been "
    f"extracted for year=2024; found {tessera_years}."
)

# SatCLIP model checkpoint/repository used.
satclip_repo = re.findall(r'repo_id\s*=\s*"([^"]*SatCLIP[^"]*)"', all_source, flags=re.I)
satclip_model = sorted(set(satclip_repo))
print("SatCLIP checkpoint repository:", satclip_model)

# Save compact evidence rather than the whole legacy notebook.
evidence_lines = [
    "TESSERA provenance evidence",
    "===========================",
    f"Explicit GeoTessera extraction year(s): {tessera_years}",
    "",
    "Relevant source lines containing GeoTessera / TESSERA year:",
]

source_lines = all_source.splitlines()
for i, line in enumerate(source_lines):
    if "sample_embeddings_at_points" in line or (
        "GeoTessera" in line and "from geotessera" in line
    ):
        start = max(0, i-2)
        end = min(len(source_lines), i+4)
        evidence_lines.extend(
            f"{j+1}: {source_lines[j]}"
            for j in range(start, end)
        )
        evidence_lines.append("")

TESSERA_PROVENANCE_EVIDENCE_PATH.write_text(
    "\n".join(evidence_lines),
    encoding="utf-8"
)
print("Saved:", TESSERA_PROVENANCE_EVIDENCE_PATH)

## 2. Read the supplier metadata for the aerial imagery

Each 25-centimetre Digimap JPG tile is accompanied by an XML file containing its flight date, completion date, resolution, supplier and tile reference. These records describe the London-wide collection used by DINOv2. A separate older 5-centimetre test order is deliberately excluded because it was not used for the final sample.

In [ ]:
# Read supplier XML metadata for the London-wide aerial tiles.
xml_files = sorted(DIGIMAP_DIR.rglob("*.xml"))
jpg_files = sorted(
    list(DIGIMAP_DIR.rglob("*.jpg"))
    + list(DIGIMAP_DIR.rglob("*.JPG"))
)

print("Digimap JPG files:", len(jpg_files))
print("Digimap XML files:", len(xml_files))

if not xml_files:
    raise FileNotFoundError("No XML supplier metadata found under Raw Data/Digimap.")


def local_name(tag):
    return tag.split("}")[-1] if "}" in tag else tag.split(":")[-1]


def parse_supplier_xml(path):
    row = {
        "xml_path": str(path),
        "xml_file": path.name,
        "batch_folder": path.parent.name,
    }
    try:
        root = ET.parse(path).getroot()
        values = {}
        for elem in root.iter():
            name = local_name(elem.tag)
            text = (elem.text or "").strip()
            if text:
                values.setdefault(name, []).append(text)

        def first(*keys):
            for key in keys:
                vals = values.get(key)
                if vals:
                    return vals[0]
            return np.nan

        row.update({
            "km_reference": first("kmReference"),
            "date_flown_raw": first("dateFlown"),
            "date_completion_raw": first("dateOfCompletion"),
            "resolution_m": pd.to_numeric(
                first("resolution"), errors="coerce"
            ),
            "created_by": first("createdBy"),
            "copyright": first("copyright"),
            "nominal_image_scale": first("nominalImageScale"),
            "correction_type": first("correctionType"),
        })
        row["parse_error"] = None
    except Exception as exc:
        row["parse_error"] = repr(exc)
    return row


aerial_meta = pd.DataFrame(
    parse_supplier_xml(p) for p in xml_files
)

aerial_meta["date_flown"] = pd.to_datetime(
    aerial_meta["date_flown_raw"],
    dayfirst=True,
    errors="coerce"
)
aerial_meta["date_completion"] = pd.to_datetime(
    aerial_meta["date_completion_raw"],
    dayfirst=True,
    errors="coerce"
)

display(aerial_meta.head())
print("XML parse errors:", aerial_meta["parse_error"].notna().sum())

print("\nResolution counts")
display(
    aerial_meta["resolution_m"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("n_tiles")
    .to_frame()
)

print("\nFlight-year counts")
flight_year_counts = (
    aerial_meta["date_flown"].dt.year
    .value_counts(dropna=False)
    .sort_index()
    .rename("n_tiles")
    .to_frame()
)
display(flight_year_counts)

print("\nFlight-date range")
print("min:", aerial_meta["date_flown"].min())
print("max:", aerial_meta["date_flown"].max())

print("\nCompletion-date range")
print("min:", aerial_meta["date_completion"].min())
print("max:", aerial_meta["date_completion"].max())

AERIAL_METADATA_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
aerial_meta.to_csv(AERIAL_METADATA_AUDIT_PATH, index=False)

valid_res = aerial_meta["resolution_m"].dropna()
resolution_values = sorted(valid_res.unique().tolist())

aerial_summary = {
    "n_jpg_files": int(len(jpg_files)),
    "n_xml_files": int(len(xml_files)),
    "n_xml_parse_errors": int(aerial_meta["parse_error"].notna().sum()),
    "resolution_values_m": [float(x) for x in resolution_values],
    "resolution_025m_share_pct": float(
        np.isclose(valid_res, 0.25).mean() * 100
    ) if len(valid_res) else None,
    "date_flown_coverage_pct": float(
        aerial_meta["date_flown"].notna().mean() * 100
    ),
    "flight_date_min": (
        str(aerial_meta["date_flown"].min())
        if aerial_meta["date_flown"].notna().any()
        else None
    ),
    "flight_date_max": (
        str(aerial_meta["date_flown"].max())
        if aerial_meta["date_flown"].notna().any()
        else None
    ),
    "flight_year_counts": {
        str(k): int(v)
        for k, v in aerial_meta["date_flown"].dt.year
        .value_counts(dropna=True).sort_index().items()
    },
    "completion_date_min": (
        str(aerial_meta["date_completion"].min())
        if aerial_meta["date_completion"].notna().any()
        else None
    ),
    "completion_date_max": (
        str(aerial_meta["date_completion"].max())
        if aerial_meta["date_completion"].notna().any()
        else None
    ),
    "created_by_values": sorted(
        aerial_meta["created_by"].dropna().unique().tolist()
    ),
}

with open(AERIAL_METADATA_SUMMARY_PATH, "w") as f:
    json.dump(aerial_summary, f, indent=2)

print("Saved:", AERIAL_METADATA_AUDIT_PATH)
print("Saved:", AERIAL_METADATA_SUMMARY_PATH)
display(pd.Series(aerial_summary, name="value"))

## 3. Link DINOv2 crops to their source-tile dates

A model crop may come from one aerial tile or a small mosaic. For every sample, this section records the earliest and latest source flight dates, the number of contributing tiles and whether the crop crosses acquisition years. These fields describe the imagery supplied to DINOv2 and are retained as diagnostic metadata rather than primary predictors.

In [ ]:
# Link every DINOv2 crop to the flight date of its contributing source tile or tiles.
import ast

crop_sources = pd.read_csv(
    AERIAL_CROP_SOURCE_INDEX_PATH,
    low_memory=False
)

# Map each imagery filename stem to supplier flight date.
meta_for_join = aerial_meta.copy()
meta_for_join["tile_stem"] = (
    meta_for_join["xml_file"]
    .astype(str)
    .str.replace(r"\.xml$", "", regex=True)
)
tile_date_map = dict(
    zip(meta_for_join["tile_stem"], meta_for_join["date_flown"])
)

def parse_tile_paths(value):
    if pd.isna(value):
        return []
    try:
        paths = json.loads(value)
    except Exception:
        try:
            paths = ast.literal_eval(value)
        except Exception:
            return []
    return list(paths)

temporal_rows = []
for row in crop_sources[
    ["sample_id", "task", "source_type", "n_tiles", "tile_paths"]
].itertuples(index=False):
    paths = parse_tile_paths(row.tile_paths)
    stems = [Path(p).stem for p in paths]
    dates = [
        tile_date_map.get(stem, pd.NaT)
        for stem in stems
    ]
    dates = [pd.Timestamp(d) for d in dates if pd.notna(d)]
    years = sorted({int(d.year) for d in dates})

    temporal_rows.append({
        "sample_id": str(row.sample_id),
        "task": row.task,
        "aerial_source_type": row.source_type,
        "aerial_n_source_tiles": (
            int(row.n_tiles)
            if pd.notna(row.n_tiles)
            else len(paths)
        ),
        "aerial_date_min": min(dates) if dates else pd.NaT,
        "aerial_date_max": max(dates) if dates else pd.NaT,
        "aerial_year_min": min(years) if years else np.nan,
        "aerial_year_max": max(years) if years else np.nan,
        "aerial_mixed_years": len(years) > 1,
        "aerial_date_span_days": (
            int((max(dates) - min(dates)).days)
            if dates else np.nan
        ),
    })

aerial_sample_time = pd.DataFrame(temporal_rows)

assert len(aerial_sample_time) == 26597
assert aerial_sample_time["sample_id"].is_unique
assert aerial_sample_time["aerial_date_min"].notna().all()

only_2021 = (
    (aerial_sample_time["aerial_year_min"] == 2021)
    & (aerial_sample_time["aerial_year_max"] == 2021)
)
only_2022 = (
    (aerial_sample_time["aerial_year_min"] == 2022)
    & (aerial_sample_time["aerial_year_max"] == 2022)
)

aerial_sample_summary = {
    "n_samples": int(len(aerial_sample_time)),
    "date_coverage_pct": float(
        aerial_sample_time["aerial_date_min"].notna().mean() * 100
    ),
    "n_2021_only": int(only_2021.sum()),
    "n_2022_only": int(only_2022.sum()),
    "n_mixed_2021_2022": int(
        aerial_sample_time["aerial_mixed_years"].sum()
    ),
    "pct_2021_only": float(only_2021.mean() * 100),
    "pct_2022_only": float(only_2022.mean() * 100),
    "pct_mixed_years": float(
        aerial_sample_time["aerial_mixed_years"].mean() * 100
    ),
    "max_within_crop_date_span_days": int(
        aerial_sample_time["aerial_date_span_days"].max()
    ),
    "note": (
        "Derived from exact source tile paths used by each DINOv2 crop "
        "and supplier dateFlown metadata."
    ),
}

aerial_sample_time.to_csv(
    AERIAL_SAMPLE_TEMPORAL_AUDIT_PATH,
    index=False
)
with open(AERIAL_SAMPLE_TEMPORAL_SUMMARY_PATH, "w") as f:
    json.dump(aerial_sample_summary, f, indent=2)

display(pd.Series(aerial_sample_summary, name="value"))
print("Saved:", AERIAL_SAMPLE_TEMPORAL_AUDIT_PATH)
print("Saved:", AERIAL_SAMPLE_TEMPORAL_SUMMARY_PATH)

## 4. Search for Street View capture dates

Date-like fields are searched in the associated shapefile, and EXIF date tags are inspected in a deterministic spread of images from the archive. ZIP and Drive modification times are not treated as capture dates because they describe file handling rather than image acquisition.

In [ ]:
# Search Street View shapefile fields and image EXIF tags for capture dates.
from PIL import Image, ExifTags
from io import BytesIO

# Locate shapefile.
shp_files = sorted((STREETVIEW_DIR / "shp").rglob("*.shp"))
if not shp_files:
    shp_files = sorted(STREETVIEW_DIR.rglob("*.shp"))

shp_date_like_cols = []
shp_columns = []
shp_path = None

if shp_files:
    import geopandas as gpd
    shp_path = shp_files[0]
    street_gdf = gpd.read_file(shp_path)
    shp_columns = street_gdf.columns.astype(str).tolist()
    shp_date_like_cols = [
        c for c in shp_columns
        if re.search(r"date|year|time|capture|survey|acqui", c, flags=re.I)
    ]

print("Street View shapefile:", shp_path)
print("Date-like shapefile columns:", shp_date_like_cols)

zip_path = STREETVIEW_DIR / "Streetviews.zip"
if not zip_path.exists():
    zip_candidates = sorted(STREETVIEW_DIR.rglob("*.zip"))
    zip_candidates = [
        p for p in zip_candidates
        if "street" in p.name.lower() or "view" in p.name.lower()
    ]
    if zip_candidates:
        zip_path = zip_candidates[0]

if not zip_path.exists():
    raise FileNotFoundError("Street View image ZIP was not found.")

date_tags = {
    306: "DateTime",
    36867: "DateTimeOriginal",
    36868: "DateTimeDigitized",
}

exif_date_values = []
n_sampled = 0
n_images_with_any_exif = 0
n_images_with_date_exif = 0
exif_errors = 0

with zipfile.ZipFile(zip_path, "r") as zf:
    image_names = sorted([
        n for n in zf.namelist()
        if n.lower().endswith((".jpg", ".jpeg", ".png"))
        and "__macosx" not in n.lower()
    ])

    n_probe = min(1000, len(image_names))
    if n_probe:
        probe_idx = np.unique(
            np.linspace(
                0, len(image_names)-1,
                num=n_probe,
                dtype=int
            )
        )
    else:
        probe_idx = []

    for idx in probe_idx:
        name = image_names[int(idx)]
        n_sampled += 1
        try:
            with zf.open(name) as fh:
                img = Image.open(fh)
                exif = img.getexif()
                if exif and len(exif):
                    n_images_with_any_exif += 1

                dates = {}
                for tag_id, label in date_tags.items():
                    value = exif.get(tag_id) if exif else None
                    if value:
                        dates[label] = str(value)

                if dates:
                    n_images_with_date_exif += 1
                    exif_date_values.append({
                        "image": name,
                        **dates
                    })
        except Exception:
            exif_errors += 1

streetview_temporal = {
    "zip_path": str(zip_path),
    "n_images_in_zip": int(len(image_names)),
    "shapefile_path": str(shp_path) if shp_path else None,
    "shapefile_date_like_columns": shp_date_like_cols,
    "n_exif_images_probed": int(n_sampled),
    "n_probed_with_any_exif": int(n_images_with_any_exif),
    "n_probed_with_date_exif": int(n_images_with_date_exif),
    "exif_probe_errors": int(exif_errors),
    "example_exif_dates": exif_date_values[:20],
    "capture_time_status": (
        "date metadata detected"
        if (shp_date_like_cols or n_images_with_date_exif)
        else "not found / unresolved in available metadata audit"
    ),
    "important_note": (
        "Drive modification times and ZIP timestamps are not treated as "
        "capture dates."
    ),
}

with open(STREETVIEW_TEMPORAL_AUDIT_PATH, "w") as f:
    json.dump(streetview_temporal, f, indent=2)

display(pd.Series(streetview_temporal, name="value"))
print("Saved:", STREETVIEW_TEMPORAL_AUDIT_PATH)

## 5. Describe the time represented by the EPC outcome

EPC is based on the most recently lodged certificate available for each property and then aggregated to postcode level. Consequently, each postcode outcome combines retained property records from a range of dates rather than representing a single observation year.

In [ ]:
# Read the retained EPC certificate-date distribution from the preparation summary.
with open(EPC_FINAL_SUMMARY, "r") as f:
    epc_audit = json.load(f)

epc_time = {
    "min_retained_record_date": epc_audit.get("epc_min_record_date"),
    "median_retained_record_date": epc_audit.get("epc_median_record_date"),
    "max_retained_record_date": epc_audit.get("epc_max_record_date"),
}
display(pd.Series(epc_time, name="EPC retained record date"))

## 6. Assemble the provenance table

For each target and representation, the table records the source, observation period, spatial unit and temporal status. The date of the environmental observation is kept conceptually separate from the period in which an encoder was trained.

In [ ]:
# Assemble one provenance record for every target and representation.
# Derive concise aerial temporal wording from the XML audit.
flight_years = sorted(
    int(k) for k in aerial_summary["flight_year_counts"].keys()
)
if flight_years:
    aerial_time = (
        str(flight_years[0])
        if len(flight_years) == 1
        else f"{flight_years[0]}–{flight_years[-1]} (tile-specific)"
    )
else:
    aerial_time = "unresolved"

streetview_time = (
    "available metadata detected; see Street View audit"
    if streetview_temporal["capture_time_status"] == "date metadata detected"
    else "not found / unresolved in available metadata audit"
)

provenance_rows = [
    {
        "item": "PTAL target",
        "role": "downstream target",
        "source": "TfL PTAL / Access Index",
        "observation_time": "2023",
        "spatial_support": "PTAL grid cell; modelling point uses cell centroid",
        "representation_or_target": "continuous Access Index / PTAL",
        "dimension": np.nan,
        "current_common_coverage": "6,597 PTAL samples",
        "temporal_status": "resolved",
        "notes": "PTAL branch was later thinned on a 500 m modelling grid.",
    },
    {
        "item": "EPC target",
        "role": "downstream target + controls",
        "source": "England/Wales EPC register extract used in project",
        "observation_time": (
            f"latest retained property records span "
            f"{epc_time['min_retained_record_date']} to "
            f"{epc_time['max_retained_record_date']}; "
            f"median {epc_time['median_retained_record_date']}"
        ),
        "spatial_support": "property records aggregated to postcode",
        "representation_or_target": "mean current energy-efficiency score",
        "dimension": np.nan,
        "current_common_coverage": "20,000 EPC postcodes",
        "temporal_status": "resolved as a multi-date register snapshot",
        "notes": "Not a single-year target.",
    },
    {
        "item": "SatCLIP",
        "role": "location representation",
        "source": "microsoft/SatCLIP-ResNet18-L40 checkpoint used by extraction code",
        "observation_time": "no sample-specific observation date (coordinate input)",
        "spatial_support": "point longitude/latitude",
        "representation_or_target": "pretrained coordinate/location embedding",
        "dimension": 256,
        "current_common_coverage": "100%",
        "temporal_status": "not applicable as an observation layer",
        "notes": "Encoder provenance is distinct from observation-time alignment.",
    },
    {
        "item": "TESSERA",
        "role": "EO representation",
        "source": "GeoTessera, sampled at project points",
        "observation_time": str(tessera_years[0]),
        "spatial_support": "pixel-level EO representation sampled at point",
        "representation_or_target": "TESSERA embedding",
        "dimension": 128,
        "current_common_coverage": "100%",
        "temporal_status": "resolved from extraction code",
        "notes": "Extraction code explicitly calls sample_embeddings_at_points(..., year=2024).",
    },
    {
        "item": "AlphaEarth 2024",
        "role": "satellite/geospatial representation",
        "source": "Imago Google Satellite Embedding V1 small-area GeoPackage",
        "observation_time": "2024",
        "spatial_support": "small-area polygon embedding inherited by sample point",
        "representation_or_target": "64-D small-area embedding",
        "dimension": 64,
        "current_common_coverage": "100% after 12 nearest-polygon boundary fallbacks",
        "temporal_status": "resolved",
        "notes": "Coarser spatial support than point/pixel/image representations.",
    },
    {
        "item": "DINOv2 aerial",
        "role": "aerial image representation",
        "source": "Getmapping / EDINA Aerial Digimap 25 cm tiles in Raw Data/Digimap",
        "observation_time": aerial_time,
        "spatial_support": (
            f"PTAL {PTAL_AERIAL_CROP_M} m crop width; "
            f"EPC {EPC_AERIAL_CROP_M} m crop width"
        ),
        "representation_or_target": "facebook/dinov2-base CLS embedding",
        "dimension": 768,
        "current_common_coverage": "100%",
        "temporal_status": (
            "resolved from supplier XML"
            if flight_years else "unresolved"
        ),
        "notes": (
            "Tile-specific flight dates are retained in aerial_25cm_metadata_audit.csv; "
            "do not substitute the separate 5 cm test-order date."
        ),
    },
    {
        "item": "Street View CLIP",
        "role": "street-level image representation",
        "source": "Street View images supplied by project supervisor",
        "observation_time": streetview_time,
        "spatial_support": (
            f"PTAL: up to k={STREETVIEW_PTAL_K} within "
            f"{STREETVIEW_PTAL_RADIUS_M} m; "
            f"EPC: up to k={STREETVIEW_EPC_K} within "
            f"{STREETVIEW_EPC_RADIUS_M} m"
        ),
        "representation_or_target": "openai/clip-vit-base-patch32, distance-weighted aggregate",
        "dimension": 512,
        "current_common_coverage": "EPC 98.02%; PTAL 86.903%",
        "temporal_status": (
            "partly/fully resolved from supplied metadata"
            if streetview_temporal["capture_time_status"] == "date metadata detected"
            else "unresolved and must be stated as a limitation"
        ),
        "notes": (
            "Image coordinates are approximated using the associated street-segment "
            "midpoint in the extraction pipeline."
        ),
    },
]

provenance = pd.DataFrame(provenance_rows)
display(provenance)

provenance.to_csv(PROVENANCE_TABLE_PATH, index=False)
print("Saved:", PROVENANCE_TABLE_PATH)

## 7. Study-time interpretation

The evidence does not support describing all targets and inputs as contemporaneous. The appropriate framing is a multi-temporal, cross-sectional multimodal comparison using the most relevant available observation period for each source. Temporal mismatch—particularly the unresolved Street View date and the broad EPC certificate range—is therefore a limitation of interpretation rather than a hidden assumption.

In [ ]:
# Record unresolved dates and the appropriate multi-temporal study framing.
unresolved_items = provenance.loc[
    provenance["temporal_status"].astype(str).str.contains(
        "unresolved", case=False, na=False
    ),
    "item"
].tolist()

temporal_summary = {
    "recommended_study_framing": (
        "latest-available cross-sectional multimodal representation "
        "comparison across London"
    ),
    "same_year_design": False,
    "ptal_year": 2023,
    "tessera_year": int(tessera_years[0]),
    "alphaearth_year": 2024,
    "epc_min_retained_record_date": epc_time["min_retained_record_date"],
    "epc_median_retained_record_date": epc_time["median_retained_record_date"],
    "epc_max_retained_record_date": epc_time["max_retained_record_date"],
    "aerial_flight_years": flight_years,
    "aerial_sample_year_distribution": aerial_sample_summary,
    "streetview_capture_time_status": streetview_temporal["capture_time_status"],
    "unresolved_temporal_items": unresolved_items,
    "interpretation": (
        "Temporal mismatch is documented explicitly; file creation or Drive "
        "modification timestamps are not used as acquisition dates."
    ),
}

with open(TEMPORAL_ALIGNMENT_SUMMARY_PATH, "w") as f:
    json.dump(temporal_summary, f, indent=2)

display(pd.Series(temporal_summary, name="value"))
print("Saved:", TEMPORAL_ALIGNMENT_SUMMARY_PATH)

## 8. Save the provenance record

The final provenance table and machine-readable summary are saved after the known dates, resolutions and unresolved items have been recorded. This provides the temporal reference used when reporting the modelling results.

In [ ]:
# Save the provenance table after confirming the key source facts.
aerial_resolution_is_025 = bool(
    aerial_summary["resolution_values_m"]
    and all(
        np.isclose(float(v), 0.25)
        for v in aerial_summary["resolution_values_m"]
    )
)

audit04 = {
    "tessera_year_verified": tessera_years == [2024],
    "n_aerial_jpg": aerial_summary["n_jpg_files"],
    "n_aerial_xml": aerial_summary["n_xml_files"],
    "aerial_xml_parse_errors": aerial_summary["n_xml_parse_errors"],
    "aerial_resolution_values_m": aerial_summary["resolution_values_m"],
    "aerial_all_reported_resolutions_are_025m": aerial_resolution_is_025,
    "aerial_flight_date_coverage_pct": aerial_summary["date_flown_coverage_pct"],
    "aerial_flight_years": flight_years,
    "aerial_sample_temporal_coverage_pct": aerial_sample_summary["date_coverage_pct"],
    "aerial_samples_2021_only": aerial_sample_summary["n_2021_only"],
    "aerial_samples_2022_only": aerial_sample_summary["n_2022_only"],
    "aerial_samples_mixed_years": aerial_sample_summary["n_mixed_2021_2022"],
    "streetview_capture_time_status": streetview_temporal["capture_time_status"],
    "same_year_design": False,
    "recommended_study_framing": temporal_summary["recommended_study_framing"],
    "unresolved_temporal_items": unresolved_items,
}

assert audit04["tessera_year_verified"], "TESSERA year verification failed."
assert audit04["n_aerial_xml"] > 0, "No aerial supplier XML was audited."
assert audit04["aerial_xml_parse_errors"] == 0, "Some aerial XML files failed to parse."
assert np.isclose(audit04["aerial_sample_temporal_coverage_pct"], 100.0), "Some DINOv2 crops lack resolved source flight dates."
assert audit04["aerial_all_reported_resolutions_are_025m"], (
    "The London-wide Digimap folder contains reported resolutions other than 0.25 m; "
    "inspect before treating it as one 25 cm source."
)

with open(PROVENANCE_AUDIT_SUMMARY_PATH, "w") as f:
    json.dump(audit04, f, indent=2)

print("04 provenance/temporal audit decision gate: PASS")
display(pd.Series(audit04, name="value"))
print("Saved:", PROVENANCE_AUDIT_SUMMARY_PATH)